## Glucose Prediction 


This assignment's focus is on predicting blood glucose. There are four parts to the assignment:


   1. Data cleaning
 
   2. Population level model
 
   3. Improving model training
 
   4. Transfer learning

In [1]:
import warnings
warnings.filterwarnings('ignore')

import sys
import os
from mads.lib.path import assets

data_cleaners = assets.find("shared_files/assignment2/data_cleaners.py")
sys.path.append(os.path.dirname(data_cleaners))
import data_cleaners as dc

find_best_hyperparameters = assets.find("shared_files/assignment2/find_best_hyperparameters.py")
sys.path.append(os.path.dirname(find_best_hyperparameters))
import find_best_hyperparameters as fbh

import pandas as pd

In [2]:
# get historical data
file_X = assets.find("assignment2/glucose_X_2d.csv")
file_y = assets.find("assignment2/glucose_y_2d.csv")
data = dc.Diabetes(file_X, file_y)

X = data.X
y = data.y

pid_to_indices = {}
for pid in X['patient_id'].unique().tolist():
    X_df = X[X['patient_id'] == pid]
    pid_to_indices[pid] = X_df.index

splits = dc.TrainSplits(X, y, combine=True, pid_to_indices=pid_to_indices)

## <font color='magenta'>Task One UnGraded</font>

## Transfer learning

In this notebook you will use an additional diabetes dataset to pre-train the same LSTM model you used in the last notebook. You'll then inspect the performance across the population of training a per-person model after pre-training the model. 

You will implement the following function in ```best_lstm_parameters_training_data_transfer```. A path has been provided for you to save the trained model to. All you need to do is to initialize your LSTM model as you have in 2b and 2c, but this time, you will it on historical data. Once the model has been trained, it should be saved to the provided path for future use in transfer learning tasks. 

**Note** in 2c we introduced a learning rate scheduler. In this notebook we will also use a learning rate scheduler, as in ```best_lstm_parameters_training_data_with_lr``` for training. As we saw before, a learning rate scheduler is a useful tool and serves to maintain stability during the training process. 

Please be patient, as the training process typically takes approximately 10 minutes to complete. 


This task is ungraded. The entire point is for you to save your model because it will be used in Task Two. 

If you do not find that the best solution is hidden size 128 then you may have made a mistake in your implementation. 

In [3]:
# Your solution goes in best_lstm_parameters_training_data_transfer in find_best_hyperparameters.py

# Once you have completed the task, please comment out the following call, so that the autograder does not retrain
# the model.
best_hidden_size = fbh.best_lstm_parameters_training_data_transfer(splits.train_df_X,
                                                splits.train_df_y, 
                                                splits.validate_df_X, 
                                                splits.validate_df_y,
                                                './best_model.pt')
#Is your best hidden size 128?

# Uncomment the line below to prepare for task 2
best_hidden_size = 128

Hidden layer size: 16
Hidden layer size: 32
Hidden layer size: 64
Hidden layer size: 128


In [4]:
print("Best hidden size:", best_hidden_size)
print("Saved checkpoint:", os.path.isfile("./best_model.pt"))

Best hidden size: 128
Saved checkpoint: True


## <font color='magenta'>Task Two</font>

## Transfer Learning

In this exercise, you will be asked to fill in the ```rmse_from_pre_training_historical_population_testing_individual_best_lstm``` function. The goal is to use the pre-trained model weights from the population of 10 historical patients and then re-train on the 10 patients we've been using so far. However, BEFORE you begin training, make sure to set the weights according to the model stored in the path below. In the transfer learning lecture video, you saw a line of pseudocode which explains how to set weights according to another model. And you saw an overview of what this is doing under the hood (11:49-13:56 or slides 30-36). Recall, in the video you saw the line of code:

```model.load_state_dict(torch.load(path_to_saved_model))```

For this to work for you, make sure `model` is the model you are currently training and make sure `path_to_saved_model` is the path referenced below. 


You are not looking for any hyper-parameters, for example you aren't evaluating the model at different values of hidden layers. You are only changing the values of the weights in the pre-existing historical model.
For example, results should look like the following (note that numbers below are correct and can be used to verify your approach)
```python
{1.0:[19.252104, 17.640997, ... , 19.940397],
 2.0:[6.85176, 6.360637, ... , 7.639355],
 ...
 10.0:[5.303737, 5.418270, ... , 5.859479]}
```



TIPS - You are not iterating over different values of the hidden layers, you will use the value found above of 128. 

In [5]:
# get present day data
file_X = assets.find("assignment2/glucose_X_2a.csv")
file_y = assets.find("assignment2/glucose_y_2a.csv")
data = dc.Diabetes(file_X, file_y)

X = data.X
y = data.y

pid_to_indices = {}
for pid in X['patient_id'].unique().tolist():
    X_df = X[X['patient_id'] == pid]
    pid_to_indices[pid] = X_df.index

splits = dc.TrainSplits(X, y, combine=True, pid_to_indices=pid_to_indices)

In [6]:
# your solution goes in 
# rmse_from_pre_training_historical_population_testing_individual_best_lstm()
# in find_best_hyperparameters.py
result_dict = fbh.rmse_from_pre_training_historical_population_testing_individual_best_lstm(best_hidden_size, 
                                                                                            './best_model.pt', 
                                                                                            splits.data_dicts)

Transfer learning, patient ID: 1.0
Transfer learning, patient ID: 2.0
Transfer learning, patient ID: 3.0
Transfer learning, patient ID: 4.0
Transfer learning, patient ID: 5.0
Transfer learning, patient ID: 6.0
Transfer learning, patient ID: 7.0
Transfer learning, patient ID: 8.0
Transfer learning, patient ID: 9.0
Transfer learning, patient ID: 10.0


In [7]:
import math

print("Imported helper:", fbh.__file__)
print("Patients:", len(result_dict))

assert isinstance(result_dict, dict)
assert len(result_dict) == 10

for pid, result in result_dict.items():
    assert isinstance(result, tuple) and len(result) == 2, f"Patient {pid}: expected a two-item tuple"
    rmse_30, horizons = result
    assert isinstance(horizons, list) and len(horizons) == 12, f"Patient {pid}: expected a list of 12 RMSEs"
    assert all(math.isfinite(value) for value in [rmse_30, *horizons]), f"Patient {pid}: non-finite RMSE"
    assert math.isclose(rmse_30, horizons[5]), f"Patient {pid}: 30-minute RMSE mismatch"

print("Patient 1 scalar type:", type(result_dict[1.0][0]).__name__)
print("All 10 patient results passed the format check.")

Imported helper: /voc/work/shared_files/assignment2/find_best_hyperparameters.py
Patients: 10
Patient 1 scalar type: float
All 10 patient results passed the format check.


In [8]:
for pid in (1.0, 2.0, 10.0):
    rmse_30, horizons = result_dict[pid]
    print(
        f"Patient {pid:.0f} | 30 min: {rmse_30:.6f} | "
        f"5 min: {horizons[0]:.6f} | 60 min: {horizons[-1]:.6f}"
    )

Patient 1 | 30 min: 19.252104 | 5 min: 17.640997 | 60 min: 19.940397
Patient 2 | 30 min: 6.851758 | 5 min: 6.360637 | 60 min: 7.639355
Patient 10 | 30 min: 5.303737 | 5 min: 5.418270 | 60 min: 5.859479


In [9]:
print(result_dict[1.0])

(19.252104448703463, [17.640997082927726, 17.95847904638077, 18.272353764113216, 18.32000637347823, 18.80145763773104, 19.252104448703463, 19.219352186997565, 19.49109995160532, 19.665774683918166, 19.58460456949988, 19.573108783600635, 19.94039697732652])


In [10]:
#hidden tests are within this cell

In [11]:
#hidden tests are within this cell

In [12]:
#hidden tests are within this cell

In [13]:
#hidden tests are within this cell

In [14]:
#hidden tests are within this cell

In [15]:
#hidden tests are within this cell

In [16]:
#hidden tests are within this cell

In [17]:
#hidden tests are within this cell

In [18]:
#hidden tests are within this cell

In [19]:
#hidden tests are within this cell

In [20]:
#hidden tests are within this cell

In [21]:
#hidden tests are within this cell

In [22]:
#hidden tests are within this cell

In [23]:
#hidden tests are within this cell

In [24]:
#hidden tests are within this cell

## IMPORTANT: Uncomment the following code when you are ready to move on to the next assignment

In [25]:
script = assets.find('copy_shared_files.sh')
!bash {script}

=== Copying shared files back to coursework ===
✅ Shared files successfully copied back to coursework.
